## Step 1: Initializing Packages and Parameters
In this step, we install and import the machine learning libraries needed to analyze the housing dataset. We also set up our file paths and seed values upfront for clean, reusable code.



In [1]:
try:
    import xgboost
except ImportError:
    import sys
    !{sys.executable} -m pip install xgboost

import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.metrics import root_mean_squared_error
from sklearn.model_selection import KFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, TargetEncoder
from xgboost import XGBRegressor


pd.set_option('display.max_columns', 100)


train_file = 'train.csv'
test_file = 'test.csv'
target_col = 'SalePrice'
id_col = 'Id'

seed_value = 42


In [2]:
import os
print("Files inside this folder:", os.listdir('.'))


Files inside this folder: ['test.csv', 'house_prices.ipynb', '.ipynb_checkpoints', 'data_description.txt', 'sample_submission.csv', 'train.csv']


## Step 2: Reading the Data
We load both datasets into memory and print their dimensions to ensure all columns and rows are present.


In [3]:
import os
print("Python is currently looking in:", os.getcwd())

train_file = 'train.csv'
test_file = 'test.csv'


housing_train = pd.read_csv(train_file)
housing_test = pd.read_csv(test_file)

print(f"Training set dimensions: {housing_train.shape}")
print(f"Testing set dimensions: {housing_test.shape}")
housing_train.head()



Python is currently looking in: /home/student/Downloads/Umurerwa Jova - house-prices-advanced-regression-techniques
Training set dimensions: (1460, 81)
Testing set dimensions: (1459, 80)


,Id,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,LotConfig,LandSlope,Neighborhood,Condition1,Condition2,BldgType,HouseStyle,OverallQual,OverallCond,YearBuilt,YearRemodAdd,RoofStyle,RoofMatl,Exterior1st,Exterior2nd,MasVnrType,MasVnrArea,ExterQual,ExterCond,Foundation,BsmtQual,BsmtCond,BsmtExposure,BsmtFinType1,BsmtFinSF1,BsmtFinType2,BsmtFinSF2,BsmtUnfSF,TotalBsmtSF,Heating,HeatingQC,CentralAir,Electrical,1stFlrSF,2ndFlrSF,LowQualFinSF,GrLivArea,BsmtFullBath,BsmtHalfBath,FullBath,HalfBath,BedroomAbvGr,KitchenAbvGr,KitchenQual,TotRmsAbvGrd,Functional,Fireplaces,FireplaceQu,GarageType,GarageYrBlt,GarageFinish,GarageCars,GarageArea,GarageQual,GarageCond,PavedDrive,WoodDeckSF,OpenPorchSF,EnclosedPorch,3SsnPorch,ScreenPorch,PoolArea,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition,SalePrice
0,1,60,RL,65.0,8450,Pave,NaN,Reg,Lvl,AllPub,Inside,Gtl,CollgCr,Norm,Norm,1Fam,2Story,7,5,2003,2003,Gable,CompShg,VinylSd,VinylSd,BrkFace,196.0,Gd,TA,PConc,Gd,TA,No,GLQ,706,Unf,0,150,856,GasA,Ex,Y,SBrkr,856,854,0,1710,1,0,2,1,3,1,Gd,8,Typ,0,NaN,Attchd,2003.0,RFn,2,548,TA,TA,Y,0,61,0,0,0,0,NaN,NaN,NaN,0,2,2008,WD,Normal,208500
1,2,20,RL,80.0,9600,Pave,NaN,Reg,Lvl,AllPub,FR2,Gtl,Veenker,Feedr,Norm,1Fam,1Story,6,8,1976,1976,Gable,CompShg,MetalSd,MetalSd,NaN,0.0,TA,TA,CBlock,Gd,TA,Gd,ALQ,978,Unf,0,284,1262,GasA,Ex,Y,SBrkr,1262,0,0,1262,0,1,2,0,3,1,TA,6,Typ,1,TA,Attchd,1976.0,RFn,2,460,TA,TA,Y,298,0,0,0,0,0,NaN,NaN,NaN,0,5,2007,WD,Normal,181500
2,3,60,RL,68.0,11250,Pave,NaN,IR1,Lvl,AllPub,Inside,Gtl,CollgCr,Norm,Norm,1Fam,2Story,7,5,2001,2002,Gable,CompShg,VinylSd,VinylSd,BrkFace,162.0,Gd,TA,PConc,Gd,TA,Mn,GLQ,486,Unf,0,434,920,GasA,Ex,Y,SBrkr,920,866,0,1786,1,0,2,1,3,1,Gd,6,Typ,1,TA,Attchd,2001.0,RFn,2,608,TA,TA,Y,0,42,0,0,0,0,NaN,NaN,NaN,0,9,2008,WD,Normal,223500
3,4,70,RL,60.0,9550,Pave,NaN,IR1,Lvl,AllPub,Corner,Gtl,Crawfor,Norm,Norm,1Fam,2Story,7,5,1915,1970,Gable,CompShg,Wd Sdng,Wd Shng,NaN,0.0,TA,TA,BrkTil,TA,Gd,No,ALQ,216,Unf,0,540,756,GasA,Gd,Y,SBrkr,961,756,0,1717,1,0,1,0,3,1,Gd,7,Typ,1,Gd,Detchd,1998.0,Unf,3,642,TA,TA,Y,0,35,272,0,0,0,NaN,NaN,NaN,0,2,2006,WD,Abnorml,140000
4,5,60,RL,84.0,14260,Pave,NaN,IR1,Lvl,AllPub,FR2,Gtl,NoRidge,Norm,Norm,1Fam,2Story,8,5,2000,2000,Gable,CompShg,VinylSd,VinylSd,BrkFace,350.0,Gd,TA,PConc,Gd,TA,Av,GLQ,655,Unf,0,490,1145,GasA,Ex,Y,SBrkr,1145,1053,0,2198,1,0,2,1,4,1,Gd,9,Typ,1,TA,Attchd,2000.0,RFn,3,836,TA,TA,Y,192,84,0,0,0,0,NaN,NaN,NaN,0,12,2008,WD,Normal,250000


## Step 3: Target Column Exploration
We analyze the skewness of our target column and apply a mathematical log transformation to stabilize the data variance.


In [4]:
target_values = housing_train[target_col]
target_log = np.log1p(target_values)

print(f"Base data skew: {target_values.skew():.2f}")
print(f"Log transformed skew: {target_log.skew():.2f}")


Base data skew: 1.88
Log transformed skew: 0.12


## Step 4: Isolating Features and Data Types
We separate our features from the target indices and group all column names into distinct categorical and numerical lists.


In [5]:
features_train = housing_train.drop(columns=[id_col, target_col])
features_test = housing_test.drop(columns=[id_col])

categorical_features = features_train.select_dtypes(include=['object', 'category']).columns.tolist()
numerical_features = features_train.select_dtypes(include=['int64', 'float64']).columns.tolist()

print(f"Total categorical columns: {len(categorical_features)}")
print(f"Total numerical columns: {len(numerical_features)}")


Total categorical columns: 43
Total numerical columns: 36


## Step 5: Data Transformation Pipelines
We create reusable data processing blocks using ColumnTransformer. This lets us test three distinct formatting styles for text data (Ordinal, One-Hot, and Target encoding) while filling in missing cells automatically.


In [6]:
numeric_imputer = SimpleImputer(strategy='median')

ord_pipeline = Pipeline(
    steps=[
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('encoder', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)),
    ]
)

ordinal_processor = ColumnTransformer(
    transformers=[
        ('num', numeric_imputer, numerical_features),
        ('cat', ord_pipeline, categorical_features),
    ]
)

onehot_pipeline = Pipeline(
    steps=[
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False)),
    ]
)

onehot_processor = ColumnTransformer(
    transformers=[
        ('num', numeric_imputer, numerical_features),
        ('cat', onehot_pipeline, categorical_features),
    ]
)

target_pipeline = Pipeline(
    steps=[
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('encoder', TargetEncoder(smooth='auto', cv=5)),
    ]
)

target_processor = ColumnTransformer(
    transformers=[
        ('num', numeric_imputer, numerical_features),
        ('cat', target_pipeline, categorical_features),
    ]
)


## Step 6: Cross-Validation Function
We configure a 5-fold cross-validation split and write an evaluation helper function to automatically calculate the mean and standard deviation of our validation error.


In [7]:
kf_splits = KFold(n_splits=5, shuffle=True, random_state=seed_value)

def compute_model_performance(model_pipe, X_input, y_target):
    error_scores = cross_val_score(
        model_pipe,
        X_input,
        y_target,
        cv=kf_splits,
        scoring='neg_root_mean_squared_error',
        n_jobs=-1
    )
    return -error_scores.mean(), error_scores.std()


## Step 7: Executing and Benchmarking Models
We define our gradient boosting models and iterate through each preprocessing strategy to score and rank them based on their log prediction error.


In [8]:
model_dict = {
    'Gradient Boosting (Sklearn)': GradientBoostingRegressor(
        n_estimators=300,
        learning_rate=0.05,
        max_depth=4,
        random_state=seed_value
    ),
    'XGBoost': XGBRegressor(
        n_estimators=300,
        learning_rate=0.05,
        max_depth=4,
        subsample=0.85,
        colsample_bytree=0.85,
        random_state=seed_value
    )
}

preprocessor_dict = {
    'Ordinal': ordinal_processor,
    'One-Hot': onehot_processor,
    'Target': target_processor
}

performance_records = []

for m_key, m_instance in model_dict.items():
    for p_key, p_instance in preprocessor_dict.items():
        execution_pipe = Pipeline(steps=[('prep', p_instance), ('model', m_instance)])
        avg_rmse, deviation_rmse = compute_model_performance(execution_pipe, features_train, target_log)
        performance_records.append({
            'Model': m_key,
            'Encoding': p_key,
            'Mean RMSE': round(avg_rmse, 4),
            'Std RMSE': round(deviation_rmse, 4)
        })

summary_table = pd.DataFrame(performance_records).sort_values(by='Mean RMSE')
summary_table


,Model,Encoding,Mean RMSE,Std RMSE
5,XGBoost,Target,0.1282,0.0140
4,XGBoost,One-Hot,0.1292,0.0191
2,Gradient Boosting (Sklearn),Target,0.1294,0.0166
3,XGBoost,Ordinal,0.1306,0.0193
1,Gradient Boosting (Sklearn),One-Hot,0.1315,0.0216
0,Gradient Boosting (Sklearn),Ordinal,0.1327,0.0217


## Step 8: Extracting Feature Importances
We fit our best-performing pipeline configuration on the full training dataset to extract and rank the top 15 features that contribute the most to the model's price predictions.


In [9]:
optimal_pipeline = Pipeline(
    steps=[
        ('prep', target_processor),
        ('model', XGBRegressor(
            n_estimators=300,
            learning_rate=0.05,
            max_depth=4,
            subsample=0.85,
            colsample_bytree=0.85,
            random_state=seed_value
        ))
    ]
)

optimal_pipeline.fit(features_train, target_log)

all_feature_names = optimal_pipeline.named_steps['prep'].get_feature_names_out()
feature_weights = optimal_pipeline.named_steps['model'].feature_importances_

weight_table = pd.DataFrame({'Feature': all_feature_names, 'Importance': feature_weights})
top_indicators = weight_table.sort_values(by='Importance', ascending=False).head(15)
top_indicators


,Feature,Importance
3,num__OverallQual,0.292585
54,cat__ExterQual,0.137293
25,num__GarageCars,0.096402
44,cat__Neighborhood,0.080351
15,num__GrLivArea,0.043424
64,cat__CentralAir,0.032788
66,cat__KitchenQual,0.032070
11,num__TotalBsmtSF,0.025534
70,cat__GarageFinish,0.018527
12,num__1stFlrSF,0.016183


## Step 9: Testing Data Predictions and Output Export
We generate house price predictions on our unseen test dataset using the optimized pipeline. Since our model was trained using log values, we convert the predictions back to regular currency values using an exponential transformation before exporting our final submission file.


In [10]:
optimal_pipeline.fit(features_train, target_log)

final_preds_log = optimal_pipeline.predict(features_test)
final_preds_dollars = np.expm1(final_preds_log)

submission_output = pd.DataFrame({
    id_col: housing_test[id_col],
    target_col: final_preds_dollars
})

submission_output.to_csv('submission.csv', index=False)

print(f"Submission matrix shape: {submission_output.shape}")
submission_output.head()


Submission matrix shape: (1459, 2)


,Id,SalePrice
0,1461,123929.84375
1,1462,158649.00000
2,1463,183485.59375
3,1464,192194.68750
4,1465,194357.31250
